In [ ]:
"""
================================================================================
SCRIPT: EXTRACT PIXEL-LEVEL GAIN AND LOSS DATA TO CSV (PER TILE)

Linear step-by-step workflow:
1. Set folders, thresholds, and output columns.
2. Read the shapefile and keep only pristine tiles (mask_artif == 0).
3. Loop through each pristine tile:
   - Find the 2025 mangrove age raster, loss age raster,
     1984 GSW baseline raster, and 2021 GSW baseline raster.
   - Select valid 2025 age pixels (1 to 41 years inclusive -> class "gain")
     and classify "land"/"sea" using the 1984 GSW baseline.
   - Select valid loss age pixels (age > 0 -> class "loss")
     and classify "land"/"sea" using the 2021 GSW baseline.
   - Convert pixel centers to WGS84 latitude/longitude (1 = sea, anything else = land).
4. Compute calendar year as 2025 - age and save <tile_id>.csv.
================================================================================
"""

from pathlib import Path
import warnings
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.crs import CRS
from rasterio.transform import rowcol, xy
from rasterio.warp import transform as warp_coords

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# 1. SETTINGS AND PATHS
# ------------------------------------------------------------------------------
REFERENCE_YEAR = 2025
MIN_MANGROVE_AGE = 1
MAX_MANGROVE_AGE = 41  # 1 to 41 years inclusive
MIN_LOSS_AGE_EXCLUSIVE = 0  # Keep all valid loss pixels with age > 0
WGS84 = CRS.from_epsg(4326)

SHP_VALIDATION_FILE = Path(
    r"D:\MANGROVES\Research_Article_Clean\Created_Data\gmwv4_mask_anthropogenic_validation.shp"
)
AGE_RASTER_FOLDER = Path(
    r"D:\MANGROVES\Research_Article_Clean\Created_Data\Mangrove_Age_2025_Rasters"
)
LOSS_RASTER_FOLDER = Path(
    r"D:\MANGROVES\Research_Article_Clean\Created_Data\Loss_Age_2025_Rasters"
)
GSW_1984_FOLDER = Path(
    r"D:\MANGROVES\Research_Article_Clean\Created_Data\GSW_1984_baseline"
)
GSW_2021_FOLDER = Path(
    r"D:\MANGROVES\Research_Article_Clean\Created_Data\GSW_2021_baseline"
)
OUTPUT_CSV_FOLDER = Path(
    r"D:\MANGROVES\Research_Article_Clean\Created_Data\CSV_Gain_Loss_Age"
)
OUTPUT_CSV_FOLDER.mkdir(parents=True, exist_ok=True)

CSV_COLUMNS = [
    "tile_id",
    "pixel_lat_WGS84",
    "pixel_lon_WGS84",
    "age",
    "year",
    "position",
    "class",
]


def find_tile_raster(folder: Path, tile_id: str, preferred_names: list[str] | None = None) -> Path | None:
    """Locate a raster matching tile_id in folder."""
    if preferred_names:
        for name in preferred_names:
            candidate = folder / name
            if candidate.exists():
                return candidate
    matches = sorted(folder.glob(f"*{tile_id}*.tif"))
    return matches[0] if matches else None


def classify_positions_from_gsw(
    x_coords: np.ndarray,
    y_coords: np.ndarray,
    raster_crs: CRS | None,
    gsw_path: Path,
) -> np.ndarray:
    """Map pixel coordinates to a GSW baseline raster and label 'sea' (== 1) or 'land'."""
    with rasterio.open(gsw_path) as gsw_src:
        gsw_data = gsw_src.read(1)
        gsw_rows, gsw_cols = gsw_data.shape
        gsw_transform = gsw_src.transform
        gsw_crs = gsw_src.crs

    if raster_crs is not None and gsw_crs is not None and raster_crs != gsw_crs:
        gx, gy = warp_coords(raster_crs, gsw_crs, x_coords.tolist(), y_coords.tolist())
        gx_arr = np.asarray(gx, dtype=float)
        gy_arr = np.asarray(gy, dtype=float)
    else:
        gx_arr = x_coords
        gy_arr = y_coords

    gsw_r, gsw_c = rowcol(gsw_transform, gx_arr, gy_arr)
    gsw_r = np.asarray(gsw_r)
    gsw_c = np.asarray(gsw_c)

    inside_gsw = (gsw_r >= 0) & (gsw_r < gsw_rows) & (gsw_c >= 0) & (gsw_c < gsw_cols)
    positions = np.full(len(x_coords), "", dtype=object)
    if np.any(inside_gsw):
        gsw_vals = gsw_data[gsw_r[inside_gsw], gsw_c[inside_gsw]]
        # In baseline raster: 1 = permanent water / sea, anything else = land
        positions[inside_gsw] = np.where(gsw_vals == 1, "sea", "land")
    return positions


# ------------------------------------------------------------------------------
# 2. LOAD PRISTINE TILES (mask_artif == 0)
# ------------------------------------------------------------------------------
shp_table = gpd.read_file(SHP_VALIDATION_FILE)
id_col = "clean_id" if "clean_id" in shp_table.columns else "tile_id"

tiles_df = pd.DataFrame({
    "tile_id": shp_table[id_col].astype(str).str.strip(),
    "mask_artif": pd.to_numeric(shp_table["mask_artif"], errors="coerce"),
})
tiles_df = tiles_df[
    (tiles_df["mask_artif"] == 0)
    & (tiles_df["tile_id"] != "")
    & (tiles_df["tile_id"].str.lower() != "nan")
].drop_duplicates(subset=["tile_id"])

tile_ids = tiles_df["tile_id"].tolist()
total_tiles = len(tile_ids)
print(f"Found {total_tiles} pristine tiles. Saving CSVs to: {OUTPUT_CSV_FOLDER}")

# ------------------------------------------------------------------------------
# 3. PROCESS EACH TILE ONE BY ONE
# ------------------------------------------------------------------------------
written_count = 0

for idx, tile_id in enumerate(tile_ids, start=1):
    # Step 3a: Locate the 2025 age raster
    age_path = find_tile_raster(
        AGE_RASTER_FOLDER, tile_id, [f"mangrove_age_2025_{tile_id}.tif"]
    )

    # Step 3b: Locate the loss age raster
    loss_path = find_tile_raster(
        LOSS_RASTER_FOLDER, tile_id, [f"loss_age_2025_{tile_id}.tif"]
    )

    # Step 3c: Locate the 1984 (for gains) and 2021 (for losses) GSW baseline rasters
    gsw_1984_path = find_tile_raster(
        GSW_1984_FOLDER,
        tile_id,
        [f"{tile_id}.tif", f"{tile_id}_baseline_water.tif"],
    )
    gsw_2021_path = find_tile_raster(
        GSW_2021_FOLDER,
        tile_id,
        [f"{tile_id}.tif", f"{tile_id}_baseline_2021_water.tif", f"{tile_id}_baseline_water.tif"],
    )

    if age_path is None or loss_path is None or gsw_1984_path is None or gsw_2021_path is None:
        print(f"⚠️ [{tile_id}] Missing one or more input rasters; skipping tile.")
        continue

    # Step 3d: Read 2025 mangrove age raster and select valid ages 1..41 ("gain")
    with rasterio.open(age_path) as age_src:
        age_data = age_src.read(1)
        age_nodata = age_src.nodata
        age_transform = age_src.transform
        age_crs = age_src.crs

    valid_age = (
        np.isfinite(age_data)
        if np.issubdtype(age_data.dtype, np.floating)
        else np.ones(age_data.shape, dtype=bool)
    )
    if age_nodata is not None and not (
        np.issubdtype(age_data.dtype, np.floating) and np.isnan(age_nodata)
    ):
        valid_age &= age_data != age_nodata

    age_int64 = age_data.astype(np.int64)
    gain_mask = valid_age & (age_int64 >= MIN_MANGROVE_AGE) & (age_int64 <= MAX_MANGROVE_AGE)

    # Step 3e: Read loss age raster and select valid ages > 0 ("loss")
    with rasterio.open(loss_path) as loss_src:
        loss_data = loss_src.read(1)
        loss_nodata = loss_src.nodata
        loss_transform = loss_src.transform
        loss_crs = loss_src.crs

    valid_loss = (
        np.isfinite(loss_data)
        if np.issubdtype(loss_data.dtype, np.floating)
        else np.ones(loss_data.shape, dtype=bool)
    )
    if loss_nodata is not None and not (
        np.issubdtype(loss_data.dtype, np.floating) and np.isnan(loss_nodata)
    ):
        valid_loss &= loss_data != loss_nodata

    loss_int64 = loss_data.astype(np.int64)
    loss_mask = valid_loss & (loss_int64 > MIN_LOSS_AGE_EXCLUSIVE)

    # Step 3f: Build rows for "gain" (using 1984 baseline) and "loss" (using 2021 baseline)
    tile_parts = []

    for mask, values_arr, transform, raster_crs, gsw_path, class_label in [
        (gain_mask, age_data, age_transform, age_crs, gsw_1984_path, "gain"),
        (loss_mask, loss_data, loss_transform, loss_crs, gsw_2021_path, "loss"),
    ]:
        rows, cols = np.where(mask)
        if len(rows) == 0:
            continue

        px_x, px_y = xy(transform, rows, cols)
        x_coords = np.asarray(px_x, dtype=float)
        y_coords = np.asarray(px_y, dtype=float)

        # Convert coordinates to WGS84 latitude and longitude
        if raster_crs is not None and raster_crs != WGS84:
            lon_wgs, lat_wgs = warp_coords(raster_crs, WGS84, x_coords.tolist(), y_coords.tolist())
            lons = np.asarray(lon_wgs, dtype=float)
            lats = np.asarray(lat_wgs, dtype=float)
        else:
            lons = x_coords
            lats = y_coords

        # Classify "land" or "sea" using 1984 baseline for gains and 2021 baseline for losses
        positions = classify_positions_from_gsw(x_coords, y_coords, raster_crs, gsw_path)

        # Cast to float64/int32 before subtracting from 2025 to avoid uint8 overflow
        raw_ages = values_arr[rows, cols].astype(np.float64)
        ages_int = np.rint(raw_ages).astype(np.int32)
        years_int = np.rint(REFERENCE_YEAR - raw_ages).astype(np.int32)

        part_df = pd.DataFrame({
            "tile_id": tile_id,
            "pixel_lat_WGS84": np.round(lats, 6),
            "pixel_lon_WGS84": np.round(lons, 6),
            "age": ages_int,
            "year": years_int,
            "position": positions,
            "class": class_label,
        })
        part_df = part_df[part_df["position"].isin(["land", "sea"])][CSV_COLUMNS]
        tile_parts.append(part_df)

    # Step 3g: Combine and write the tile's CSV file
    if tile_parts:
        out_df = pd.concat(tile_parts, ignore_index=True)
    else:
        out_df = pd.DataFrame(columns=CSV_COLUMNS)

    out_csv = OUTPUT_CSV_FOLDER / f"{tile_id}.csv"
    out_df.to_csv(out_csv, index=False)
    written_count += 1

    if idx % 25 == 0 or idx == total_tiles:
        print(f"Progress: {idx}/{total_tiles} tiles processed ({written_count} CSVs written)")

print(f"Done! Wrote {written_count} CSV files in {OUTPUT_CSV_FOLDER}.")

Found 1261 pristine tiles. Saving CSVs to: D:\MANGROVES\Research_Article_Clean\Created_Data\CSV_Gain_Loss_Age
Progress: 25/1261 tiles processed (25 CSVs written)
Progress: 50/1261 tiles processed (50 CSVs written)
